# Entrenamiento del detector de placas (Semana 3)

Proyecto ALPR Portería — Institución Educativa Consejo Municipal La Estrella.

**Por qué en Google Colab y no en el portátil:** entrenar una red neuronal implica hacer millones de operaciones matemáticas repetidas muchas veces (una por cada imagen, muchas veces). Una GPU (tarjeta gráfica) hace esas operaciones en paralelo y es 10-20 veces más rápida que un procesador normal (CPU) para este trabajo. Colab da acceso gratuito a una GPU; sin ella, este mismo entrenamiento tomaría horas en vez de 20-30 minutos.

**Antes de empezar:** sube a tu Google Drive el dataset generado con `src/preparar_dataset_yolo.py` (la carpeta `data/dataset/` completa, comprimida en un .zip llamado `dataset.zip`).

## 1. Activar GPU

Antes de correr cualquier celda: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware → GPU`.

In [ ]:
!nvidia-smi

Si el comando anterior muestra una tabla con el nombre de una GPU (por ejemplo "Tesla T4"), la GPU está activa. Si da error, vuelve al paso 1.

## 2. Montar Google Drive y descomprimir el dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ajusta esta ruta a donde subiste dataset.zip dentro de tu Drive
RUTA_ZIP_EN_DRIVE = '/content/drive/MyDrive/alpr-porteria/dataset.zip'

# OJO: dataset.zip ya trae una carpeta "dataset/" adentro (así lo arma
# shutil.make_archive en preparar_dataset_yolo.py), así que se descomprime
# en /content directamente y no en /content/dataset — si no, queda
# duplicado como /content/dataset/dataset/images/... y el entrenamiento
# falla con "images not found, missing path .../images/val".
!unzip -q "$RUTA_ZIP_EN_DRIVE" -d /content
!find /content/dataset -maxdepth 2

import os
assert os.path.isdir('/content/dataset/images/val'), (
    'No quedó en /content/dataset/images/val. Revisa con la celda de arriba '
    'dónde quedaron los archivos y ajusta las rutas de este notebook si tu '
    'dataset.zip tiene una estructura distinta.'
)
print('Estructura correcta: /content/dataset/images/val existe.')

El `dataset.yaml` generado por `preparar_dataset_yolo.py` trae rutas absolutas del computador donde se generó, así que hay que reescribirlo para que apunte a `/content/dataset` (la ruta dentro de Colab).

In [ ]:
contenido_yaml = f"""path: /content/dataset
train: images/train
val: images/val
nc: 1
names: ['placa']
"""
with open('/content/dataset/dataset.yaml', 'w') as f:
    f.write(contenido_yaml)
print(contenido_yaml)

## 3. Instalar Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 4. Qué es una época y qué es la función de pérdida

**Época (epoch):** una pasada completa por TODAS las imágenes de entrenamiento. Entrenar 50 épocas significa que el modelo ve cada una de las imágenes de entrenamiento 50 veces, ajustando un poco sus parámetros internos cada vez. Muy pocas épocas y el modelo no alcanza a aprender el patrón (le llamamos *subajuste*); demasiadas y el modelo empieza a "memorizar" las imágenes de entrenamiento en vez de aprender el patrón general (*sobreajuste*), lo cual se nota porque deja de mejorar en las imágenes de validación aunque siga mejorando en las de entrenamiento.

**Función de pérdida (loss):** un número que mide qué tan mal lo está haciendo el modelo en un momento dado. Entrenar es, literalmente, ajustar los parámetros del modelo para que este número baje. YOLO reporta varias pérdidas por separado:
- `box_loss`: qué tan mal ubicadas/dimensionadas están las cajas predichas respecto a las reales.
- `cls_loss`: qué tan mal clasificada está cada detección (en nuestro caso, solo hay una clase, así que esta pérdida es menos crítica).
- `dfl_loss`: una pérdida auxiliar que ayuda a que los bordes de la caja sean más precisos.

**Cómo leer la gráfica de entrenamiento:** las curvas de pérdida deben bajar y luego aplanarse. Si la pérdida de entrenamiento sigue bajando pero la de validación empieza a subir, es señal de sobreajuste: el modelo está memorizando en vez de generalizar.

## 5. Entrenar

In [ ]:
from ultralytics import YOLO

modelo = YOLO('yolo11n.pt')  # partimos del modelo pequeño preentrenado, no desde cero

# epochs subió de 50 a 75: el dataset de esta vuelta es más difícil de
# aprender que el anterior (variedad de escala + fondos reales + ejemplos
# negativos), así que necesita algunas pasadas más para converger bien.
resultados = modelo.train(
    data='/content/dataset/dataset.yaml',
    epochs=75,
    imgsz=640,
    batch=16,
    project='/content/entrenamiento_placas',
    name='detector_placas',
)

## 6. Ver las gráficas de pérdida y las métricas de validación

In [ ]:
from IPython.display import Image, display

display(Image(filename='/content/entrenamiento_placas/detector_placas/results.png'))

Además de las curvas de pérdida, fíjense en dos métricas clave que aparecen en la gráfica:

- **mAP50**: qué tan bien detecta el modelo cuando se exige que la caja predicha se superponga al menos un 50% con la caja real. Es la métrica estándar para "¿el modelo encuentra la placa o no?".
- **precision / recall**: precision = de las placas que el modelo dice haber encontrado, cuántas eran realmente placas; recall = de todas las placas que había, cuántas encontró el modelo.

Estas tres cifras (mAP50, precision, recall) son las que deben poder explicar ante el jurado.

## 7. Descargar el modelo entrenado

In [ ]:
from google.colab import files

ruta_best = '/content/entrenamiento_placas/detector_placas/weights/best.pt'
files.download(ruta_best)

print('Descarga el archivo y guárdalo como modelos/best.pt en el proyecto (en tu portátil).')